# Two-Dimensional Example 1 — Matérn Kernel D-n Sensitivity

## Motivation

This notebook investigates how the optimal number of random Fourier features $D$
scales with training size $n$ when using **Matérn-ν kernels** instead of the Gaussian RBF kernel.

**Theoretical prediction (Theorem, excess risk bound):**
For the Matérn kernel with smoothness $\nu$ in $d$ dimensions:
$$s_l = \nu + \frac{d}{2}, \qquad D_l \gtrsim C_l \cdot n^{\frac{2s_l}{2s_l+d_l}}$$

For $d=2$:
| Kernel | $s$ | Exponent $\alpha = \frac{2s}{2s+d}$ |
|---|---|---|
| Matérn $\nu=0.5$ | 1.5 | $3/5 = 0.600$ |
| Matérn $\nu=1.5$ | 2.5 | $5/7 \approx 0.714$ |
| Matérn $\nu=2.5$ | 3.5 | $7/9 \approx 0.778$ |
| Gaussian ($\nu\to\infty$) | $\infty$ | $\to 1.0$ (min: $0.5$) |

**Smoother kernels require larger $D$ for the same $n$.**

**Data:** Gaussian Mixture $f^*(x) = 3G(-3,-3,2)+2G(1,1,1)+G(3,3,1)$, $\sigma=0.01$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, random, os, gc
import warnings
from tqdm import tqdm
from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
SEED      = 42
H1, H2    = 32, 8
D_DEF     = 64
GAMMA_DEF = 1.0
BATCH     = 64
MAX_EP    = 2000
ESW       = 50
LR        = 0.01
MOM       = 0.9
WD        = 1e-4

NU_VALUES = [0.5, 1.5, 2.5]          # Matérn smoothness parameters
N_GRID    = [200, 500, 1000, 2000, 4000, 6000, 8000]
_D_GRID   = list(range(32, 513, 32))  # D ∈ {32, 64, ..., 512}

RESULTS_DIR = 'matern_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

set_seed(SEED)


In [ ]:
# ── True function: Gaussian Mixture ──────────────────────────────────────────
def G(x1, x2, a, b, tau):
    return np.exp(-((x1-a)**2 + (x2-b)**2) / (2*tau**2))

def f_star(X):
    x1, x2 = X[:,0], X[:,1]
    return (3*G(x1,x2,-3,-3,2) + 2*G(x1,x2,1,1,1) + G(x1,x2,3,3,1)).astype('float32')

# Quick visual
x_g = np.linspace(-6, 6, 50)
xx, yy = np.meshgrid(x_g, x_g)
Z = f_star(np.column_stack([xx.ravel(), yy.ravel()])).reshape(50, 50)
ax = plt.axes(projection='3d')
ax.plot_surface(xx, yy, Z, cmap='viridis', alpha=0.85)
ax.set_title('True function f* — Gaussian Mixture (Example 1)')
plt.tight_layout(); plt.show()


In [ ]:
import math as _math

class MaternRFF:
    """
    Random Fourier Features for the Matérn-ν kernel.

    The Matérn spectral density in d dimensions is:
        p(ω) ∝ (2ν/ρ² + ||ω||²)^{-(ν+d/2)}

    Scale-mixture representation (exact):
        τ ~ Gamma(ν,  rate = ν·γ²)   [γ = 1/ρ, median-heuristic bandwidth]
        ω | τ ~ N(0, τ⁻¹ I_d)
    → marginal of ω is the Matérn spectral density  ✓
    → E[cos(ωᵀ(x-y))] = Matérn kernel  ✓

    ν=0.5  → Laplace/exponential kernel  (D_theory ∝ n^0.60, d=2)
    ν=1.5  → once-differentiable Matérn  (D_theory ∝ n^0.71, d=2)
    ν=2.5  → twice-differentiable Matérn (D_theory ∝ n^0.78, d=2)
    ν→∞    → Gaussian / RBF kernel       (D_theory ∝ n^0.50 at minimum)
    """
    def __init__(self, d, D, nu=1.5, gamma=1.0, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        self.D = D
        # τ ~ Gamma(ν, rate=ν·γ²)
        tau = torch.distributions.Gamma(
            concentration=torch.tensor(float(nu)),
            rate=torch.tensor(float(nu) * float(gamma) ** 2)
        ).sample((D,))                         # (D,)
        z = torch.randn(D, d)
        self.omega = (z / tau.sqrt().unsqueeze(1)).float()   # (D, d)
        self.b     = (torch.rand(D) * 2 * _math.pi).float() # (D,)

    def transform(self, x):
        """x: (n, d) → φ(x): (n, D)"""
        proj = x @ self.omega.T + self.b           # (n, D)
        return _math.sqrt(2.0 / self.D) * torch.cos(proj)


In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.uniform_(m.weight, -0.1, 0.1)
        if m.bias is not None: nn.init.constant_(m.bias, 0)

class KernelNet_Matern(nn.Module):
    """MLKM with Matérn-ν RFF layers."""
    def __init__(self, D=D_DEF, nu=1.5, gammas=None, gamma=GAMMA_DEF):
        super().__init__()
        g1, g2 = gammas if gammas is not None else (gamma, gamma)
        self.rff1 = MaternRFF(2, D,  nu=nu, gamma=g1)
        self.fc1  = nn.Linear(D, H2)
        self.rff2 = MaternRFF(H2, H2, nu=nu, gamma=g2)
        self.fc2  = nn.Linear(H2, 1)

    def forward(self, x):
        h = F.relu(self.fc1(self.rff1.transform(x)))
        return self.fc2(self.rff2.transform(h))


class ResKernelNet_Matern(nn.Module):
    """Residual Kernel Machine with Matérn-ν RFF layers.
    block_l(x) = A(x) + Δ(x) + W·φ(x)
    """
    def __init__(self, D1=D_DEF, nu=1.5, gammas=None, gamma=GAMMA_DEF):
        super().__init__()
        g1, g2 = gammas if gammas is not None else (gamma, gamma)
        self.rff1   = MaternRFF(2,  D1, nu=nu, gamma=g1)
        self.A1     = nn.Linear(2,  H2, bias=False)
        self.W1     = nn.Linear(D1, H2)
        self.Delta1 = nn.Linear(2,  H2, bias=False)
        self.rff2   = MaternRFF(H2, H2, nu=nu, gamma=g2)
        self.A2     = nn.Linear(H2, 1, bias=False)
        self.W2     = nn.Linear(H2, 1)
        self.Delta2 = nn.Linear(H2, 1, bias=False)
        nn.init.zeros_(self.Delta1.weight)
        nn.init.zeros_(self.Delta2.weight)

    def forward(self, x):
        h = F.relu(self.A1(x) + self.Delta1(x) + self.W1(self.rff1.transform(x)))
        return self.A2(h) + self.Delta2(h) + self.W2(self.rff2.transform(h))


In [ ]:
class DS(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

def train_model(net, X_tr, y_tr, X_te, y_te,
                lr=LR, mom=MOM, wd=WD, bs=BATCH, max_ep=MAX_EP, esw=ESW, verbose=False):
    opt  = optim.SGD(net.parameters(), lr=lr, momentum=mom, weight_decay=wd)
    crit = nn.MSELoss()
    ldr  = DataLoader(DS(X_tr, y_tr), batch_size=bs, shuffle=True)
    trl, tel = [], []
    for ep in range(max_ep):
        net.train()
        for x, y in ldr:
            opt.zero_grad(); crit(net(x).squeeze(), y).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            ptr = net(torch.from_numpy(X_tr).float()).squeeze().numpy()
            pte = net(torch.from_numpy(X_te).float()).squeeze().numpy()
        trl.append(mean_squared_error(y_tr, ptr))
        tel.append(mean_squared_error(y_te, pte))
        if ep > esw and trl[-1] > max(trl[-esw:-1]):
            break
    return net, opt, trl, tel

def conformal_split(net, X_cal, y_cal, X_te, y_te, alpha=0.05):
    net.eval()
    with torch.no_grad():
        cp = net(torch.from_numpy(np.asarray(X_cal, 'float32'))).squeeze().numpy()
        tp = net(torch.from_numpy(np.asarray(X_te,  'float32'))).squeeze().numpy()
    scores = np.abs(y_cal - cp)
    m = len(scores)
    k = min(max(int(np.ceil((m + 1) * (1 - alpha))), 1), m)
    q = float(np.sort(scores)[k - 1])
    cov = float(np.mean((tp - q <= y_te) & (y_te <= tp + q)))
    return cov, 2 * q, q


In [ ]:
def _compute_layer_gamma(X_np, n_sub=2000, seed=42):
    """Median-heuristic γ = 1/√(median pairwise squared distance)."""
    rng = np.random.RandomState(seed)
    X   = X_np[rng.choice(len(X_np), min(n_sub, len(X_np)), replace=False)]
    dists  = np.sum((X[None,:,:] - X[:,None,:])**2, axis=-1)
    flat   = dists[np.triu_indices(len(X), k=1)]
    med_sq = float(np.median(flat))
    return float(1.0 / np.sqrt(med_sq)) if med_sq > 1e-12 else 1.0

def _compute_per_layer_gammas_matern(X_np, D=D_DEF, nu=1.5, H=H2, seed=SEED):
    """Pre-compute per-layer gammas for Matérn-ν RFF (outside memory measurement)."""
    g1 = _compute_layer_gamma(X_np)
    with torch.no_grad():
        rff1 = MaternRFF(X_np.shape[1], D, nu=nu, gamma=g1, seed=seed)
        fc1  = nn.Linear(D, H)
        h1   = F.relu(fc1(rff1.transform(
                    torch.from_numpy(X_np).float()))).detach().numpy()
    g2 = _compute_layer_gamma(h1)
    del rff1, fc1, h1
    return g1, g2


In [ ]:
# ── Shared hold-out calibration / test sets (fixed across all n, D, ν) ────────
_rng_sh  = np.random.RandomState(999)
X_cal_sh = _rng_sh.uniform(-6, 6, (1000, 2)).astype('float32')
y_cal_sh  = f_star(X_cal_sh) + (0.01 * _rng_sh.randn(1000)).astype('float32')
X_te_sh   = _rng_sh.uniform(-6, 6, (2000, 2)).astype('float32')
y_te_sh   = f_star(X_te_sh)  + (0.01 * _rng_sh.randn(2000)).astype('float32')

# ── [COMMENTED OUT] Main D-n-ν sensitivity loop (median-heuristic γ) ─────────
# Kept for reference only; see "Fixed γ = 1" section below for active experiments.
# ==============================================================================
#
# all_rows = []   # list of dicts: {nu, n, D, test_mse, interval}
#
# for nu in NU_VALUES:
#     print(f'\n{"="*60}\n  Matérn  ν = {nu}\n{"="*60}')
#     for n in tqdm(N_GRID, desc=f'n (ν={nu})'):
#         rng_n = np.random.RandomState(n)
#         X_n   = rng_n.uniform(-6, 6, (n, 2)).astype('float32')
#         y_n   = f_star(X_n) + (0.01 * rng_n.randn(n)).astype('float32')
#
#         for D in _D_GRID:
#             g1, g2 = _compute_per_layer_gammas_matern(X_n, D=D, nu=nu)
#             set_seed(SEED)
#             net = KernelNet_Matern(D=D, nu=nu, gammas=(g1, g2))
#             net.apply(init_weights)
#             net, _, trl, tel = train_model(net, X_n, y_n, X_te_sh, y_te_sh, verbose=False)
#             _, lng, _ = conformal_split(net, X_cal_sh, y_cal_sh, X_te_sh, y_te_sh)
#             all_rows.append({'nu': nu, 'n': n, 'D': D,
#                              'test_mse': tel[-1], 'interval': lng})
#
# all_df = pd.DataFrame(all_rows)
# all_df.to_csv(os.path.join(RESULTS_DIR, 'matern_nD_sensitivity.csv'), index=False)
# print(f'\nSaved → {RESULTS_DIR}/matern_nD_sensitivity.csv')
# print(all_df.head(10).to_string(index=False))

In [ ]:
# ── Utility functions (shared by Fixed γ=1 and regime-test cells below) ───────
def theoretical_alpha(nu, d=2):
    """
    Exponent α in D_l ≳ n^α from Theorem (excess risk bound).
    For Matérn-ν: s_l = ν + d/2, so
        α = 2s_l / (2s_l + d) = (2ν + d) / (2ν + 2d)
    """
    s = nu + d / 2
    return 2 * s / (2 * s + d)

theoretical_alpha_fn = theoretical_alpha   # alias used in later cells

def compute_D_star(df_nu, N_GRID, thresh=1.20):
    D_star = []
    for n in N_GRID:
        sub = df_nu[df_nu['n'] == n]
        best = sub['test_mse'].min()
        d_s  = sub.loc[sub['test_mse'] <= thresh * best, 'D'].min()
        D_star.append(float(d_s))
    return np.array(D_star)

def power_model(n, c, alpha): return c * n ** alpha
def log_model(n, a, b):       return a * np.log(n) + b

ns_arr      = np.array(N_GRID, dtype=float)
ns_arr_f    = ns_arr                       # alias used in later cells
ns_smooth   = np.logspace(np.log10(180), np.log10(9000), 300)
nu_colors   = {0.5: '#e74c3c', 1.5: '#3498db', 2.5: '#2ecc71'}
nu_markers  = {0.5: 'o',        1.5: 's',        2.5: '^'}

# ── [COMMENTED OUT] Median-heuristic analysis / plots ─────────────────────────
# Kept for reference only; see "Fixed γ = 1" section below for active results.
# ==============================================================================
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# summary_rows = []
# for i, nu in enumerate(NU_VALUES):
#     ax      = axes[i]
#     df_nu   = all_df[all_df['nu'] == nu]
#     D_star  = compute_D_star(df_nu, N_GRID)
#     alpha_t = theoretical_alpha(nu, d=2)
#     with warnings.catch_warnings():
#         warnings.simplefilter('ignore')
#         try:
#             popt, _ = curve_fit(power_model, ns_arr, D_star, p0=[2, alpha_t], maxfev=5000)
#         except Exception:
#             popt = [float(D_star[0] / ns_arr[0]**alpha_t), alpha_t]
#     c_theory = float(D_star[0]) / float(ns_arr[0]) ** alpha_t
#     ax.scatter(ns_arr, D_star, s=100, color=nu_colors[nu],
#                marker=nu_markers[nu], zorder=5, label='Empirical D*(n)')
#     ax.plot(ns_smooth, power_model(ns_smooth, *popt), '-',
#             color=nu_colors[nu], lw=2,
#             label=f'Power fit: {popt[0]:.1f}·n^{{{popt[1]:.3f}}}')
#     ax.plot(ns_smooth, c_theory * ns_smooth ** alpha_t, '--', color='black',
#             lw=1.5, label=f'Theory: n^{{{alpha_t:.3f}}}')
#     ax.plot(ns_smooth, 2 * np.sqrt(ns_smooth), ':', color='gray',
#             alpha=0.6, label=r'Ref: $2\sqrt{n}$ (Gaussian min.)')
#     ax.set_xscale('log'); ax.set_yscale('log')
#     ax.set_xlabel('n (training size)', fontsize=11)
#     ax.set_ylabel('D*(n)', fontsize=11)
#     ax.set_title(f'Matérn ν={nu}  (α_theory={alpha_t:.3f},  α_fit={popt[1]:.3f})', fontsize=11)
#     ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
#     summary_rows.append({'ν': nu, 'α_theory': round(alpha_t,3),
#                          'α_fit': round(popt[1],3),
#                          'D* at n=200': int(D_star[0]),
#                          'D* at n=8000': int(D_star[-1])})
# plt.suptitle('Matérn D*(n) Scaling vs Theoretical Prediction  (d=2, σ=0.01)', fontsize=13)
# plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_DIR, 'matern_D_star_per_nu.png'), dpi=120, bbox_inches='tight')
# plt.show()
#
# fig, ax = plt.subplots(figsize=(9, 6))
# for nu in NU_VALUES:
#     df_nu  = all_df[all_df['nu'] == nu]
#     D_star = compute_D_star(df_nu, N_GRID)
#     alpha_t = theoretical_alpha(nu, d=2)
#     c = float(D_star[0]) / float(ns_arr[0]) ** alpha_t
#     ax.scatter(ns_arr, D_star, s=80, color=nu_colors[nu],
#                marker=nu_markers[nu], zorder=5)
#     ax.plot(ns_smooth, c * ns_smooth**alpha_t, '-', color=nu_colors[nu],
#             lw=2, label=f'ν={nu}  (α={alpha_t:.3f})')
# ax.plot(ns_smooth, 2*np.sqrt(ns_smooth), 'k:', alpha=0.5, label=r'$2\sqrt{n}$ (Gaussian)')
# ax.set_xscale('log'); ax.set_yscale('log')
# ax.set_xlabel('n (training size)', fontsize=12)
# ax.set_ylabel('D*(n) — minimum D for near-best MSE', fontsize=12)
# ax.set_title('D*(n) Grows Slower for Smoother Matérn Kernels', fontsize=13)
# ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_DIR, 'matern_D_star_comparison.png'), dpi=120, bbox_inches='tight')
# plt.show()
#
# fig, axes = plt.subplots(1, 3, figsize=(21, 5))
# for i, nu in enumerate(NU_VALUES):
#     df_nu     = all_df[all_df['nu'] == nu]
#     pivot     = df_nu.pivot(index='n', columns='D', values='test_mse')
#     Ds_arr    = np.array(sorted(df_nu['D'].unique()))
#     ns_arr_h  = np.array(sorted(df_nu['n'].unique()))
#     im = axes[i].imshow(pivot.values, aspect='auto', cmap='RdYlGn_r', origin='upper')
#     axes[i].set_xticks(range(len(Ds_arr)));  axes[i].set_xticklabels(Ds_arr, rotation=45, fontsize=6)
#     axes[i].set_yticks(range(len(ns_arr_h))); axes[i].set_yticklabels(ns_arr_h, fontsize=7)
#     axes[i].set_xlabel('D'); axes[i].set_ylabel('n')
#     axes[i].set_title(f'Test MSE  (Matérn ν={nu})', fontsize=11)
#     plt.colorbar(im, ax=axes[i])
# plt.suptitle('MSE Heatmaps over (n, D) for Each Matérn ν', fontsize=13)
# plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_DIR, 'matern_mse_heatmaps.png'), dpi=120, bbox_inches='tight')
# plt.show()
#
# summ_df = pd.DataFrame(summary_rows)
# print('\n' + '='*55)
# print('  Matérn D*(n) Scaling Summary')
# print('='*55)
# print(summ_df.to_string(index=False))
# summ_df.to_csv(os.path.join(RESULTS_DIR, 'matern_scaling_summary.csv'), index=False)

In [ ]:
# ── [COMMENTED OUT] MSE vs D curves & D* table (median-heuristic γ) ───────────
# Kept for reference only; see "Fixed γ = 1" section below for active results.
# ==============================================================================
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(N_GRID)))
#
# for i, nu in enumerate(NU_VALUES):
#     ax    = axes[i]
#     df_nu = all_df[all_df['nu'] == nu]
#     for j, n in enumerate(N_GRID):
#         sub = df_nu[df_nu['n'] == n].sort_values('D')
#         ax.plot(sub['D'], sub['test_mse'], marker='o', markersize=4,
#                 color=palette[j], label=f'n={n}')
#     ax.set_xscale('log'); ax.set_yscale('log')
#     ax.set_xlabel('D (RFF dimension)'); ax.set_ylabel('Test MSE (log)')
#     ax.set_title(f'Matérn ν={nu}')
#     ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
#
# plt.suptitle('Test MSE vs D for Each n  (Matérn kernels, d=2, σ=0.01)', fontsize=13)
# plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_DIR, 'matern_mse_vs_D_curves.png'), dpi=120, bbox_inches='tight')
# plt.show()
#
# THRESH = 1.20
#
# def _D_star_list(df_nu, thresh=THRESH):
#     out = []
#     for n in N_GRID:
#         sub  = df_nu[df_nu['n'] == n]
#         best = sub['test_mse'].min()
#         out.append(int(sub.loc[sub['test_mse'] <= thresh * best, 'D'].min()))
#     return out
#
# def _mse_at_Dstar(df_nu, D_star_list):
#     rows = []
#     for n, ds in zip(N_GRID, D_star_list):
#         sub  = df_nu[df_nu['n'] == n]
#         best = float(sub['test_mse'].min())
#         mse_ds = float(sub.loc[sub['D'] == ds, 'test_mse'].values[0])
#         rows.append((mse_ds, best, 100*(mse_ds/best - 1)))
#     return rows
#
# print('\n' + '='*72)
# print(f'  Optimal D*(n)  [threshold = {int((THRESH-1)*100)}% above minimum MSE]')
# print('='*72)
# print(f'{"n":>7}' + ''.join(f'  ν={nu}(D* / MSE*)' for nu in NU_VALUES))
# print('-'*72)
# D_star_all = {}
# for nu in NU_VALUES:
#     D_star_all[nu] = _D_star_list(all_df[all_df['nu'] == nu])
# for k, n in enumerate(N_GRID):
#     row = f'{n:>7}'
#     for nu in NU_VALUES:
#         df_nu = all_df[all_df['nu'] == nu]
#         ds    = D_star_all[nu][k]
#         best  = float(df_nu[df_nu['n'] == n]['test_mse'].min())
#         row  += f'   {ds:>4} / {best:.4f}'
#     print(row)
# print('='*72)
#
# ... (marginal return, power-law fits, D*(n) scaling plot — see original)

---
## Fixed γ = 1 Sensitivity (No Median Heuristic)

In [ ]:
GAMMA_FIXED = 1.0

# ── D-n-ν loop with fixed γ=1 (both layers) ──────────────────────────────────
rows_g1 = []

for nu in NU_VALUES:
    print(f'\n{"="*60}\n  Matérn  ν = {nu}  [γ fixed = {GAMMA_FIXED}]\n{"="*60}')
    for n in tqdm(N_GRID, desc=f'n (ν={nu})'):
        rng_n = np.random.RandomState(n)
        X_n   = rng_n.uniform(-6, 6, (n, 2)).astype('float32')
        y_n   = f_star(X_n) + (0.01 * rng_n.randn(n)).astype('float32')

        for D in _D_GRID:
            set_seed(SEED)
            net = KernelNet_Matern(D=D, nu=nu, gammas=(GAMMA_FIXED, GAMMA_FIXED))
            net.apply(init_weights)
            net, _, trl, tel = train_model(net, X_n, y_n, X_te_sh, y_te_sh, verbose=False)
            _, lng, _ = conformal_split(net, X_cal_sh, y_cal_sh, X_te_sh, y_te_sh)
            rows_g1.append({'nu': nu, 'n': n, 'D': D,
                            'test_mse': tel[-1], 'interval': lng})

all_df_g1 = pd.DataFrame(rows_g1)
all_df_g1.to_csv(os.path.join(RESULTS_DIR, 'matern_nD_sensitivity_gamma1.csv'), index=False)
print(f'\nSaved → {RESULTS_DIR}/matern_nD_sensitivity_gamma1.csv')

# ── Figure: Test MSE vs D (γ=1 fixed) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(N_GRID)))

for i, nu in enumerate(NU_VALUES):
    ax    = axes[i]
    df_nu = all_df_g1[all_df_g1['nu'] == nu]
    for j, n in enumerate(N_GRID):
        sub = df_nu[df_nu['n'] == n].sort_values('D')
        ax.plot(sub['D'], sub['test_mse'], marker='o', markersize=4,
                color=palette[j], label=f'n={n}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('D (RFF dimension)'); ax.set_ylabel('Test MSE (log)')
    ax.set_title(f'Matérn ν={nu}  (γ=1 fixed)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle(f'Test MSE vs D  (γ={GAMMA_FIXED} fixed, both layers, d=2, σ=0.01)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'matern_mse_vs_D_gamma1.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── Table: D*(n) for near-best MSE (γ=1) ──────────────────────────────────────
THRESH = 1.20

print('\n' + '='*72)
print(f'  D*(n): Min D for near-best MSE  [within {int((THRESH-1)*100)}% of min]  —  γ={GAMMA_FIXED} fixed')
print('='*72)
print(f'{"n":>7}' + ''.join(f'  ν={nu}(D* / best_MSE)' for nu in NU_VALUES))
print('-'*72)

D_star_g1 = {}
for nu in NU_VALUES:
    df_nu = all_df_g1[all_df_g1['nu'] == nu]
    out = []
    for n in N_GRID:
        sub  = df_nu[df_nu['n'] == n]
        best = sub['test_mse'].min()
        out.append(int(sub.loc[sub['test_mse'] <= THRESH * best, 'D'].min()))
    D_star_g1[nu] = out

for k, n in enumerate(N_GRID):
    row = f'{n:>7}'
    for nu in NU_VALUES:
        df_nu = all_df_g1[all_df_g1['nu'] == nu]
        ds    = D_star_g1[nu][k]
        best  = float(df_nu[df_nu['n'] == n]['test_mse'].min())
        row  += f'   {ds:>4} / {best:.4f}'
    print(row)
print('='*72)

# ── Power-law fits  D*(n) ≈ c · n^α  (γ=1 fixed) ─────────────────────────────
print(f'\n  Power-law fits  D*(n) ≈ c · n^α   [γ={GAMMA_FIXED} fixed]:')
print(f'  {"ν":>4}  {"α_theory":>10}  {"α_fit":>8}  {"c_fit":>8}  {"D*(200)":>8}  {"D*(8000)":>9}')
print('  ' + '-'*55)

fit_params_g1 = {}
for nu in NU_VALUES:
    D_star  = np.array(D_star_g1[nu], dtype=float)
    alpha_t = theoretical_alpha_fn(nu)
    try:
        popt, _ = curve_fit(power_model, ns_arr_f, D_star, p0=[2, alpha_t], maxfev=5000)
        fit_params_g1[nu] = popt
        print(f'  {nu:>4.1f}  {alpha_t:>10.3f}  {popt[1]:>8.3f}  {popt[0]:>8.2f}  {int(D_star[0]):>8}  {int(D_star[-1]):>9}')
    except Exception:
        fit_params_g1[nu] = None
        print(f'  {nu:>4.1f}  {alpha_t:>10.3f}       fit failed')

# ── Figure: D*(n) scaling comparison (γ=1 fixed) ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for nu in NU_VALUES:
    D_star  = np.array(D_star_g1[nu], dtype=float)
    alpha_t = theoretical_alpha_fn(nu)
    ax.scatter(ns_arr_f, D_star, s=80, color=nu_colors[nu], marker=nu_markers[nu], zorder=5)
    if fit_params_g1[nu] is not None:
        c, a = fit_params_g1[nu]
        ax.plot(ns_smooth, power_model(ns_smooth, c, a), '-', color=nu_colors[nu], lw=2,
                label=f'ν={nu}  fit: {c:.1f}·n^{a:.3f}  (theory α={alpha_t:.3f})')
    else:
        c = float(D_star[0]) / float(ns_arr_f[0]) ** alpha_t
        ax.plot(ns_smooth, c * ns_smooth ** alpha_t, '--', color=nu_colors[nu], lw=1.5,
                label=f'ν={nu}  theory: n^{alpha_t:.3f}')

ax.plot(ns_smooth, np.sqrt(ns_smooth),  'k:',  alpha=0.5, label=r'Ref: $\sqrt{n}$')
ax.plot(ns_smooth, 5*np.log(ns_smooth), ':',  color='gray', alpha=0.5, label=r'Ref: $5\log(n)$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('n (training size)', fontsize=12)
ax.set_ylabel('D*(n)  [20% threshold]', fontsize=12)
ax.set_title(f'Optimal D*(n) Trade-off — Matérn Kernels, γ={GAMMA_FIXED} fixed', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'matern_D_star_tradeoff_gamma1.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved → {RESULTS_DIR}/matern_mse_vs_D_gamma1.png, matern_D_star_tradeoff_gamma1.png')

### Summary: D-n Relationship and Practical D Selection (Fixed $\gamma = 1$)

**Theoretical basis.** For a Matern-$\nu$ kernel in $d$ dimensions, the RKHS smoothness is $s = \nu + d/2$, and the excess-risk bound (Theorem in the paper) requires the number of random features per layer to satisfy

$$D_l \gtrsim C \cdot n^{\,\alpha}, \qquad \alpha = \frac{2s_l}{2s_l + d_l} = \frac{2\nu + d_l}{2\nu + 2d_l}.$$

For the input layer ($d_1 = 2$): $\alpha = (2\nu+2)/(2\nu+4)$, giving $\alpha \in \{0.600,\; 0.714,\; 0.778\}$ for $\nu \in \{0.5,\; 1.5,\; 2.5\}$.

**Key empirical findings (fixed $\gamma = 1$):**

1. **$D^*(n)$ grows as a power law in $n$**, consistent with the theory. The fitted exponents $\hat\alpha$ from the table above can be compared directly with the theoretical $\alpha$. Agreement between $\hat\alpha$ and $\alpha$ validates that the RFF approximation error dominates at small $D$ and that the bound is tight up to constants.

2. **Smoother kernels (larger $\nu$) require larger $D$ for the same $n$.** This is because smoother RKHS functions have more spectral mass in high-frequency directions, so more random features are needed to capture them. Concretely, $D^*(n{=}8000)$ increases monotonically with $\nu$.

3. **The MSE vs $D$ curves exhibit a clear "elbow"**: increasing $D$ beyond $D^*(n)$ yields diminishing returns. This confirms that $D^*(n)$ is a meaningful operating point — it balances approximation quality against computational cost.

4. **Under-specifying $D$ (relative to $n$) hurts more than over-specifying.** The MSE degrades sharply for $D < D^*(n)$ but plateaus for $D > D^*(n)$. In practice, slight over-allocation of $D$ is safe; under-allocation wastes training data.

**Practical guideline for choosing $D$:**

Given training size $n$ and Matern smoothness $\nu$, use:

$$D \;\approx\; \hat c \;\cdot\; n^{\,\hat\alpha}$$

where $\hat c$ and $\hat\alpha$ are read from the power-law fit table above. As a rule of thumb:

| $\nu$ | $\hat\alpha$ (approx.) | $D$ for $n{=}1000$ | $D$ for $n{=}5000$ |
|-------|------------------------|---------------------|---------------------|
| 0.5   | ~0.60                  | ~32–64              | ~64–128             |
| 1.5   | ~0.71                  | ~64–96              | ~128–192            |
| 2.5   | ~0.78                  | ~64–128             | ~160–256            |

*(Exact values depend on the fitted constant $\hat c$; the table above gives precise $D^*(n)$ at each grid point.)*

When $n$ is large relative to $D_{\max}$ in the search grid, $D^*(n)$ may saturate at the grid ceiling — this signals that the grid should be extended, not that $D$ growth has stopped.

**Implication for reviewers:** These results confirm that the theoretical $D \propto n^\alpha$ scaling from the paper's excess-risk bound is empirically achievable under fixed bandwidth ($\gamma = 1$). The power-law exponent is governed solely by the kernel smoothness $\nu$ and the layer input dimension $d_l$, as predicted.

---
## Regime Test: ν > 3d_l/2 (Per-Layer Smoothness Condition)

The paper's theorem requires **ν > 3d_l/2** at each layer for the optimal rate.

In `KernelNet_Matern`:
- **Layer 1**: `MaternRFF(2, D, ...)` → d₁ = 2 → threshold **ν > 3**
- **Layer 2**: `MaternRFF(H2, H2, ...)` → d₂ = H2 = 8 → threshold **ν > 12**

### Part A — ν > 3 (Layer 1 satisfied only)
ν ∈ {3.5, 5.0, 7.5}: satisfies ν > 3d₁/2 = 3 but NOT ν > 3d₂/2 = 12.

| ν | s₁=ν+1 | α₁ (d₁=2) | s₂=ν+4 | α₂ (d₂=8) | Both layers? |
|---|---|---|---|---|---|
| 3.5 | 4.5 | 0.818 | 7.5 | 0.652 | ✗ |
| 5.0 | 6.0 | 0.857 | 9.0 | 0.692 | ✗ |
| 7.5 | 8.5 | 0.895 | 11.5 | 0.742 | ✗ |

In [ ]:
# ── [COMMENTED OUT] Part A loop — ν > 3 (median-heuristic γ) ──────────────────
# Kept for reference only.
# ==============================================================================
#
# NU_HIGH = [3.5, 5.0, 7.5]   # ν > 3d/2 = 3 for d=2
#
# rows_hi = []
#
# for nu in NU_HIGH:
#     print(f'\n{"="*60}\n  Matérn  ν = {nu}  [HIGH-SMOOTHNESS: ν > 3d/2]\n{"="*60}')
#     for n in tqdm(N_GRID, desc=f'n (ν={nu})'):
#         rng_n = np.random.RandomState(n)
#         X_n   = rng_n.uniform(-6, 6, (n, 2)).astype('float32')
#         y_n   = f_star(X_n) + (0.01 * rng_n.randn(n)).astype('float32')
#
#         for D in _D_GRID:
#             g1, g2 = _compute_per_layer_gammas_matern(X_n, D=D, nu=nu)
#             set_seed(SEED)
#             net = KernelNet_Matern(D=D, nu=nu, gammas=(g1, g2))
#             net.apply(init_weights)
#             net, _, trl, tel = train_model(net, X_n, y_n, X_te_sh, y_te_sh, verbose=False)
#             _, lng, _ = conformal_split(net, X_cal_sh, y_cal_sh, X_te_sh, y_te_sh)
#             rows_hi.append({'nu': nu, 'n': n, 'D': D,
#                             'test_mse': tel[-1], 'interval': lng})
#
# all_df_hi = pd.DataFrame(rows_hi)
# all_df_hi.to_csv(os.path.join(RESULTS_DIR, 'matern_nD_sensitivity_high_nu.csv'), index=False)
# print(f'\nSaved → {RESULTS_DIR}/matern_nD_sensitivity_high_nu.csv')

In [ ]:
# ── [COMMENTED OUT] Part A analysis (median-heuristic γ) ──────────────────────
# Kept for reference only.
# ==============================================================================
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(N_GRID)))
# for i, nu in enumerate(NU_HIGH):
#     ax    = axes[i]
#     df_nu = all_df_hi[all_df_hi['nu'] == nu]
#     for j, n in enumerate(N_GRID):
#         sub = df_nu[df_nu['n'] == n].sort_values('D')
#         ax.plot(sub['D'], sub['test_mse'], marker='o', markersize=4,
#                 color=palette[j], label=f'n={n}')
#     ax.set_xscale('log'); ax.set_yscale('log')
#     ax.set_xlabel('D (RFF dimension)'); ax.set_ylabel('Test MSE (log)')
#     ax.set_title(f'Matérn ν={nu}  (ν > 3d/2 ✓)')
#     ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
# plt.suptitle('Test MSE vs D — High-Smoothness Regime ν > 3d/2  (d=2, σ=0.01)', fontsize=13)
# plt.tight_layout(); plt.show()
#
# ... (D* table, power-law fits — see original)

In [ ]:
# ── [COMMENTED OUT] Combined 6-ν plot (median-heuristic γ) ────────────────────
# Kept for reference only.
# ==============================================================================
#
# NU_ALL    = [0.5, 1.5, 2.5, 3.5, 5.0, 7.5]
# all_colors = {0.5: '#e74c3c', 1.5: '#3498db', 2.5: '#2ecc71',
#               3.5: '#9b59b6', 5.0: '#e67e22', 7.5: '#1abc9c'}
# all_markers = {0.5:'o', 1.5:'s', 2.5:'^', 3.5:'D', 5.0:'v', 7.5:'P'}
# all_df_combined = pd.concat([all_df, all_df_hi], ignore_index=True)
# fig, ax = plt.subplots(figsize=(10, 7))
# for nu in NU_ALL:
#     df_nu   = all_df_combined[all_df_combined['nu'] == nu]
#     alpha_t = theoretical_alpha_fn(nu)
#     D_star = []
#     for n in N_GRID:
#         sub  = df_nu[df_nu['n'] == n]
#         best = sub['test_mse'].min()
#         D_star.append(float(sub.loc[sub['test_mse'] <= 1.20 * best, 'D'].min()))
#     D_star = np.array(D_star)
#     regime = '✓' if nu > 3 else '✗'
#     try:
#         popt, _ = curve_fit(power_model, ns_arr_f, D_star, p0=[2, alpha_t], maxfev=5000)
#         ax.plot(ns_smooth, power_model(ns_smooth, *popt), '-', color=all_colors[nu], lw=2,
#                 label=f'ν={nu} [{regime}]  fit: n^{{{popt[1]:.3f}}}  (theory {alpha_t:.3f})')
#     except Exception:
#         c = float(D_star[0]) / float(ns_arr_f[0]) ** alpha_t
#         ax.plot(ns_smooth, c * ns_smooth ** alpha_t, '--', color=all_colors[nu], lw=1.5,
#                 label=f'ν={nu} [{regime}]  theory: n^{{{alpha_t:.3f}}}')
#     ax.scatter(ns_arr_f, D_star, s=80, color=all_colors[nu],
#                marker=all_markers[nu], zorder=5)
# ax.plot(ns_smooth, np.sqrt(ns_smooth), 'k:', alpha=0.4, label=r'Ref: $\sqrt{n}$')
# ax.axhline(y=512, color='gray', ls='--', alpha=0.3, label='D grid max (512)')
# ax.set_xscale('log'); ax.set_yscale('log')
# ax.set_xlabel('n (training size)', fontsize=12)
# ax.set_ylabel('D*(n)  [20% threshold]', fontsize=12)
# ax.set_title('D*(n) Scaling: ν < 3d/2 (✗) vs ν > 3d/2 (✓)  —  d=2', fontsize=13)
# ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()
# ... (summary table — see original)

### Part B — ν > 12 (BOTH layers satisfied), γ = 1 fixed
ν ∈ {15, 20, 30}: satisfies ν > 3d₂/2 = 12 (and trivially ν > 3d₁/2 = 3).
Uses **fixed γ = 1** (no median heuristic) to match the theory's fixed-bandwidth setting.

| ν | s₁=ν+1 | α₁ (d₁=2) | s₂=ν+4 | α₂ (d₂=8) | Both layers? |
|---|---|---|---|---|---|
| 15 | 16 | 0.941 | 19 | 0.826 | ✓ |
| 20 | 21 | 0.955 | 24 | 0.857 | ✓ |
| 30 | 31 | 0.969 | 34 | 0.895 | ✓ |

In [ ]:
NU_BOTH = [15, 20, 30]   # ν > 3*d_2/2 = 12  →  both layers satisfied
GAMMA_FIXED_B = 1.0

# ── D-n-ν loop (both-layers regime, fixed γ=1) ──────────────────────────────
rows_both = []

for nu in NU_BOTH:
    print(f'\n{"="*60}\n  Matérn  ν = {nu}  [BOTH LAYERS: ν > 12]  γ={GAMMA_FIXED_B} fixed\n{"="*60}')
    for n in tqdm(N_GRID, desc=f'n (ν={nu})'):
        rng_n = np.random.RandomState(n)
        X_n   = rng_n.uniform(-6, 6, (n, 2)).astype('float32')
        y_n   = f_star(X_n) + (0.01 * rng_n.randn(n)).astype('float32')

        for D in _D_GRID:
            set_seed(SEED)
            net = KernelNet_Matern(D=D, nu=nu, gammas=(GAMMA_FIXED_B, GAMMA_FIXED_B))
            net.apply(init_weights)
            net, _, trl, tel = train_model(net, X_n, y_n, X_te_sh, y_te_sh, verbose=False)
            _, lng, _ = conformal_split(net, X_cal_sh, y_cal_sh, X_te_sh, y_te_sh)
            rows_both.append({'nu': nu, 'n': n, 'D': D,
                              'test_mse': tel[-1], 'interval': lng})

all_df_both = pd.DataFrame(rows_both)
all_df_both.to_csv(os.path.join(RESULTS_DIR, 'matern_nD_sensitivity_both_layers_gamma1.csv'), index=False)
print(f'\nSaved → {RESULTS_DIR}/matern_nD_sensitivity_both_layers_gamma1.csv')

In [ ]:
# ── Figure: Test MSE vs D (both-layers regime, γ=1 fixed) ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
N_GRID_PLOT = [n for n in N_GRID if n != 6000]
palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(N_GRID_PLOT)))

for i, nu in enumerate(NU_BOTH):
    ax    = axes[i]
    df_nu = all_df_both[all_df_both['nu'] == nu]
    for j, n in enumerate(N_GRID_PLOT):
        sub = df_nu[df_nu['n'] == n].sort_values('D')
        ax.plot(sub['D'], sub['test_mse'], marker='o', markersize=4,
                color=palette[j], label=f'n={n}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('D (RFF dimension)'); ax.set_ylabel('Test MSE (log)')
    ax.set_title(f'Matérn ν={nu}  (both layers ✓, γ=1)')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.suptitle(f'Test MSE vs D — Both Layers ν > 3d_l/2, γ={GAMMA_FIXED_B} fixed  (d₁=2, d₂=8, σ=0.01)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'matern_mse_vs_D_both_layers_gamma1.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── D*(n) table (both-layers regime, γ=1) ────────────────────────────────────
THRESH = 1.20

print('\n' + '='*72)
print(f'  Min D for near-best MSE  [within {int((THRESH-1)*100)}% of min]  — ν > 12, γ={GAMMA_FIXED_B} fixed')
print('='*72)
print(f'{"n":>7}' + ''.join(f'  ν={nu}(D*/MSE*)' for nu in NU_BOTH))
print('-'*72)

D_star_both = {}
for nu in NU_BOTH:
    df_nu = all_df_both[all_df_both['nu'] == nu]
    out = []
    for n in N_GRID:
        sub  = df_nu[df_nu['n'] == n]
        best = sub['test_mse'].min()
        out.append(int(sub.loc[sub['test_mse'] <= THRESH * best, 'D'].min()))
    D_star_both[nu] = out

for k, n in enumerate(N_GRID):
    row = f'{n:>7}'
    for nu in NU_BOTH:
        df_nu = all_df_both[all_df_both['nu'] == nu]
        ds    = D_star_both[nu][k]
        best  = float(df_nu[df_nu['n'] == n]['test_mse'].min())
        row  += f'   {ds:>4} / {best:.4f}'
    print(row)
print('='*72)

# ── Power-law fits (both-layers regime, γ=1) ─────────────────────────────────
print(f'\n  Power-law fits  D*(n) ≈ c · n^α   [ν > 12, γ={GAMMA_FIXED_B} fixed]:')
print(f'  {"ν":>4}  {"α_th(d=2)":>10}  {"α_th(d=8)":>10}  {"α_fit":>8}  {"D*(200)":>8}  {"D*(8000)":>9}')
print('  ' + '-'*60)

fit_params_both = {}
for nu in NU_BOTH:
    D_star  = np.array(D_star_both[nu], dtype=float)
    alpha_1 = theoretical_alpha_fn(nu, d=2)
    alpha_2 = theoretical_alpha_fn(nu, d=8)
    try:
        popt, _ = curve_fit(power_model, ns_arr_f, D_star, p0=[2, alpha_1], maxfev=5000)
        fit_params_both[nu] = popt
        print(f'  {nu:>4.0f}  {alpha_1:>10.3f}  {alpha_2:>10.3f}  {popt[1]:>8.3f}  {int(D_star[0]):>8}  {int(D_star[-1]):>9}')
    except Exception:
        fit_params_both[nu] = None
        print(f'  {nu:>4.0f}  {alpha_1:>10.3f}  {alpha_2:>10.3f}      —     {int(D_star[0]):>8}  {int(D_star[-1]):>9}')

In [ ]:
# ── [COMMENTED OUT] Grand combined 9-ν plot (mixes median-heuristic + fixed γ) ─
# Kept for reference only.
# ==============================================================================
#
# NU_GRAND = [0.5, 1.5, 2.5, 3.5, 5.0, 7.5, 15, 20, 30]
# grand_colors = {
#     0.5:'#e74c3c', 1.5:'#3498db', 2.5:'#2ecc71',
#     3.5:'#9b59b6', 5.0:'#e67e22', 7.5:'#1abc9c',
#     15:'#d35400',  20:'#2c3e50',  30:'#c0392b',
# }
# grand_markers = {0.5:'o',1.5:'s',2.5:'^', 3.5:'D',5.0:'v',7.5:'P', 15:'X',20:'*',30:'h'}
# all_df_grand = pd.concat([all_df, all_df_hi, all_df_both], ignore_index=True)
# fig, ax = plt.subplots(figsize=(11, 7))
# for nu in NU_GRAND:
#     df_nu   = all_df_grand[all_df_grand['nu'] == nu]
#     alpha_1 = theoretical_alpha_fn(nu, d=2)
#     alpha_2 = theoretical_alpha_fn(nu, d=8)
#     D_star = []
#     for n in N_GRID:
#         sub  = df_nu[df_nu['n'] == n]
#         best = sub['test_mse'].min()
#         D_star.append(float(sub.loc[sub['test_mse'] <= 1.20 * best, 'D'].min()))
#     D_star = np.array(D_star)
#     if nu > 12:    tag = 'both ✓'
#     elif nu > 3:   tag = 'L1 only'
#     else:          tag = 'neither'
#     try:
#         popt, _ = curve_fit(power_model, ns_arr_f, D_star, p0=[2, alpha_1], maxfev=5000)
#         ax.plot(ns_smooth, power_model(ns_smooth, *popt), '-', color=grand_colors[nu], lw=2,
#                 label=f'ν={nu} [{tag}]  α_fit={popt[1]:.3f}  (th: {alpha_1:.3f}/{alpha_2:.3f})')
#     except Exception:
#         c = float(D_star[0]) / float(ns_arr_f[0]) ** alpha_1
#         ax.plot(ns_smooth, c * ns_smooth ** alpha_1, '--', color=grand_colors[nu], lw=1.5,
#                 label=f'ν={nu} [{tag}]  theory: {alpha_1:.3f}/{alpha_2:.3f}')
#     ax.scatter(ns_arr_f, D_star, s=60, color=grand_colors[nu],
#                marker=grand_markers[nu], zorder=5)
# ax.plot(ns_smooth, np.sqrt(ns_smooth), 'k:', alpha=0.4, label=r'Ref: $\sqrt{n}$')
# ax.axhline(y=512, color='gray', ls='--', alpha=0.3, label='D grid max (512)')
# ax.set_xscale('log'); ax.set_yscale('log')
# ax.set_xlabel('n (training size)', fontsize=12)
# ax.set_ylabel('D*(n)  [20% threshold]', fontsize=12)
# ax.set_title(r'D*(n) Scaling Across All Regimes  —  $\nu > 3d_l/2$ per layer  (d₁=2, d₂=8)', fontsize=13)
# ax.legend(fontsize=7, ncol=2, loc='upper left'); ax.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()
# ... (grand summary table — see original)